# 伪随机数的预测：用 Z3 反推 `rand()` 的内部状态

## 这一章讲什么

伪随机数看上去杂乱无章，但只要生成的方法已知，此后出现的值都可以预测出来。这一章以 Visual C++ 的 `rand()` 为例，用 Z3 反推它的内部状态。由于生成式完全公开，因此反推所需的全部信息都藏在输出之中，下面先列出要点。

- `rand()` 由线性同余生成器(LCG)生成
- 拿到几个输出以后，可以用 Z3 反推内部状态(种子)
- 种子一旦求出，下一个值也就可以预测

全章只需要两个工具：`BitVec`(定长的比特串)与 `URem`(无符号取余)，二者分别对应状态表示和取余运算。

## 线性同余生成器(LCG)

线性同余生成器(Linear Congruential Generator, LCG)是生成伪随机数的经典方法之一，其下一个状态由下面这条递推式给出。

$$
X_{n+1} = \left(A \times X_n + B\right) \bmod M
$$

其中 $A$、$B$、$M$ 都是常数，一般要满足下面列出的几条条件，这些条件保证序列不会很快退化。

- $M > A$
- $M > B$
- $A > 0$
- $B \geq 0$

当 $M = 2^{32}$ 时，这个式子表示的正是 32 位整数的乘法与加法，再把溢出的部分丢弃。

## Visual C++ 的 `rand()`

Visual C++ 的 `rand()` 采用的正是线性同余这条路，其源码很短，只维护了一个静态变量，如下所示。

```c
static long holdrand = 1L;

int __cdecl rand(void)
{
    return (((holdrand = holdrand * 214013L + 2531011L) >> 16) & 0x7FFF);
}

void __cdecl srand(unsigned int seed)
{
    holdrand = (long)seed;
}
```

它只做两件事。

1. 把内部状态 `holdrand` 往前推进一步

$$
\text{holdrand} = (\text{holdrand} \times 214013 + 2531011) \bmod 2^{32}
$$

2. 取出高 16 位，作为 0〜32767(`0x7FFF`)的整数返回

输出得到的只是内部状态的一部分，因此从输出倒推状态就变成了一个问题：信息量是否足够。下面我们写一个行为完全相同的 Python 类，用它产生后面要用到的实验数据。

In [1]:
class PseudoRandom:
    """和 Visual C++ 的 rand() 行为相同的伪随机数"""

    def __init__(self, seed=1):
        self._holdrand = seed

    def srand(self, seed: int):
        self._holdrand = seed

    def rand(self) -> int:
        self._holdrand = (self._holdrand * 214013 + 2531011) & 0xFFFFFFFF
        return (self._holdrand >> 16) & 0x7FFF

## 先生成 8 个随机数

种子取 `1792`，连续生成 8 个 `rand() % 100`(取值 0〜99)，而第 9 个值暂时保留下来，最后用来核对答案。

In [2]:
rand = PseudoRandom(1792)
numbers = [rand.rand() % 100 for _ in range(8)]
print(numbers)
print(rand.rand() % 100)

[90, 79, 97, 7, 20, 77, 17, 89]
16


## 用 Z3 反推种子

`numbers` 里只有 8 个数，第 9 个能不能猜出来，这是本节要回答的问题。内部状态是 32 位，因此用一个 `BitVec('seed', 32)` 变量来表示它，然后把 `rand()` 的那几行计算原封不动地写成约束。

1. `states[0]` 表示“种子走了一步以后的状态”，`states[1]` 表示再往后一步的状态，如此依次排下去
2. 相邻两个状态必须满足 `state_{i+1} = state_i * a + b`(都在 32 位以内循环，乘出来的溢出部分自动丢弃)
3. 每个状态取出的 `(state >> 16) & 0x7FFF` 除以 `mod` 取余，结果要与观测到的 `numbers` 一致

由于只看得到除以 `mod` 之后的余数，因此状态本身未必能唯一确定，不过“能够生成同一串数的种子”总是找得到的。

In [3]:
from z3 import *


def find_seed_and_next(numbers, mod, a=214013, b=2531011):
    """从观测到的输出序列，求出种子和下一个输出"""
    seed = BitVec('seed', 32)
    states = [BitVec(f's_{i}', 32) for i in range(len(numbers) + 1)]
    outputs = [URem((s >> 16) & 0x7FFF, mod) for s in states]

    solver = Solver()

    # 1. 状态的转移（LCG）
    for prev, cur in zip([seed] + states[:-1], states):
        solver.add(prev * a + b == cur)

    # 2. 和观测到的输出一致
    for out, observed in zip(outputs, numbers):
        solver.add(out == observed)

    if solver.check() != sat:
        return None

    model = solver.model()
    return model[seed].as_long(), model.eval(outputs[-1]).as_long()


seed_val, next_val = find_seed_and_next(numbers, mod=100)
print(f'seed = {seed_val}')
print(f'next = {next_val}')

seed = 1792
next = 16


原来的种子 `1792` 被反推出来了，下一个值算出来是 `16`，而真正的第 9 个数也是 `16`(可以翻到上面的格子核对输出)，下面用这个种子把数列重新生成一遍。

In [4]:
rand = PseudoRandom(seed_val)
[rand.rand() % 100 for _ in range(9)]

[90, 79, 97, 7, 20, 77, 17, 89, 16]

9 个数一个一个排出来，与原来完全相同，预测成功。

不过 `mod 100` 这一步截掉了一部分信息，因此满足条件的种子未必只有一个。这道题里 `1792` 和 `2147485440` 都能过关，两个种子给出的数列也完全一样。由此可知，输出得到的信息越少，候选种子就越多。

## 换个玩法：让骰子连出 10 个 6

骰子是用 `rand() % 6 + 1` 掷出来的，下面找一个种子，使它连着十次都掷出 6。要掷出 6，只需 `rand() % 6 == 5`，因此把观测值直接传成 `[5] * 10` 即可。

In [5]:
seed_val, next_val = find_seed_and_next([5] * 10, mod=6)
print(seed_val)

rand = PseudoRandom(seed_val)
[rand.rand() % 6 for _ in range(11)]

3689596556


[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0]

连着十次都是 `5`(也就是骰子掷出 6)的种子，已经找到了。

不过要注意，放进条件里的只有前 10 次，第 11 次并没有约束，因此它不一定仍然是 `5`(这一次运行的结果是 `0`)。如果要对准“11 次连出 6”，把那十一项 `[5] * 11` 全部放进条件即可，此外满足条件的种子不止一个，每运行一次都可能遇到不同的种子。

## `BitVec` 和 `URem`

`BitVec` 是 Z3 中定长比特串的类型，其运算按位进行，超出规定长度的部分自动丢弃。

- `BitVec('x', 32)` 可以当作 32 位整数使用
- 乘法与加法的溢出部分一律丢弃(与 C 的无符号整数相同)
- `&`, `|`, `^`, `>>`, `<<` 这些位运算都可以使用

`URem(a, b)` 表示无符号取余，即 `a % b`。由于 `BitVec` 没有符号位，因此取余必须调用 `URem()`，不能直接使用 `%`，有了这两样工具，`rand()` 的代码照着抄写即可，例如下面这张 C 代码与 Z3 表达式的对照表。

| C 的代码 | Z3 的表达式 |
|---|---|
| `holdrand * 214013 + 2531011` | `prev * 214013 + 2531011` |
| 32 位溢出的部分丢弃 | `BitVec` 的计算(自动) |
| `holdrand >> 16` | `s >> 16` |
| `& 0x7FFF` | `& 0x7FFF` |
| `% mod` | `URem(..., mod)` |

## 小结

- 生成式已知，就可以从输出反推内部状态，做法是把生成式原样写进 `BitVec`
- 输出只露出状态的一部分时，种子候选可能不止一个
- 有了反推出来的种子，下一个值就可以算出
- 像“连续 10 次都是 6”这样的要求，也只是把观测值传进去，由 Z3 去寻找种子